In [ ]:
!pip install segmentation-models-pytorch albumentations opencv-python matplotlib scikit-learn pycocotools roboflow -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.2/249.2 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 34.5 MB/s eta 0:00:00


In [ ]:
import os, cv2, random, json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR

import albumentations as A
from albumentations.pytorch import ToTensorV2

import segmentation_models_pytorch as smp
from pycocotools.coco import COCO
from pycocotools import mask as maskUtils

print("All imports OK")

All imports OK


In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="rZo2uFSwFj9aHJPSaF2u")
project = rf.workspace("capstone-project-qiyij").project("combined-dataset-seg-full-vkwbr")
version = project.version(1)
dataset = version.download("coco-segmentation")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Combined-Dataset-Seg-Full-1 in coco-segmentation:: 100%|██████████| 9640/9640 [00:02<00:00, 3583.19it/s]


In [ ]:
TRAIN_DIR  = "/content/Combined-Dataset-Seg-Full-1/train"
VALID_DIR  = "/content/Combined-Dataset-Seg-Full-1/valid"
TEST_DIR   = "/content/Combined-Dataset-Seg-Full-1/test"
TRAIN_COCO = f"{TRAIN_DIR}/_annotations.coco.json"
VALID_COCO = f"{VALID_DIR}/_annotations.coco.json"
TEST_COCO  = f"{TEST_DIR}/_annotations.coco.json"

IMG_SIZE   = 512
BATCH_SIZE = 4
EPOCHS     = 50
LR         = 3e-4
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {DEVICE}  |  IMG_SIZE: {IMG_SIZE}  |  EPOCHS: {EPOCHS}")

Device: cuda  |  IMG_SIZE: 512  |  EPOCHS: 50


In [ ]:
def apply_clahe(image_rgb: np.ndarray) -> np.ndarray:
    """
    Apply CLAHE in LAB space so all 3 channels are preserved.
    This avoids the baseline bug of fake-RGB from a single gray channel.
    """
    lab   = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2LAB)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])   # enhance L channel only
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

In [ ]:
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),

    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.Rotate(limit=20, p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=0,
                       border_mode=cv2.BORDER_CONSTANT, p=0.4),

    A.ElasticTransform(alpha=80, sigma=8, p=0.3),
    A.GridDistortion(num_steps=5, distort_limit=0.2, p=0.3),

    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.4),
    A.GaussNoise(var_limit=(5.0, 25.0), p=0.3),     # simulate X-ray noise
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.3),  # random CLAHE augment

    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
Argument(s) 'var_limit' are not valid for transform GaussNoise


In [ ]:
class CocoFractureDataset(Dataset):
    def __init__(self, image_dir, annotation_path, transform=None, use_clahe=True):
        self.image_dir  = image_dir
        self.coco       = COCO(annotation_path)
        self.image_ids  = list(self.coco.imgs.keys())
        self.transform  = transform
        self.use_clahe  = use_clahe

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id   = self.image_ids[idx]
        image_info = self.coco.loadImgs(image_id)[0]

        image = cv2.imread(os.path.join(self.image_dir, image_info["file_name"]))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.use_clahe:
            image = apply_clahe(image)

        h, w  = image_info["height"], image_info["width"]
        mask  = np.zeros((h, w), dtype=np.uint8)

        ann_ids = self.coco.getAnnIds(imgIds=image_id)
        for ann in self.coco.loadAnns(ann_ids):
            mask = np.maximum(mask, self.coco.annToMask(ann))

        if self.transform:
            aug   = self.transform(image=image, mask=mask.astype("float32"))
            image = aug["image"]
            mask  = aug["mask"].unsqueeze(0).float()

        return image, mask


train_dataset = CocoFractureDataset(TRAIN_DIR, TRAIN_COCO, train_transform, use_clahe=True)
val_dataset   = CocoFractureDataset(VALID_DIR, VALID_COCO, val_transform,   use_clahe=True)
test_dataset  = CocoFractureDataset(TEST_DIR,  TEST_COCO,  val_transform,   use_clahe=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

loading annotations into memory...
Done (t=0.24s)
creating index...
index created!
loading annotations into memory...
Done (t=0.03s)
creating index...
index created!
loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
Train: 6744 | Val: 1448 | Test: 1440


In [ ]:
model = smp.UnetPlusPlus(
    encoder_name     = "efficientnet-b4",
    encoder_weights  = "imagenet",
    in_channels      = 3,
    classes          = 1,
    activation       = None,
    decoder_channels = (256, 128, 64, 32, 16),
    decoder_attention_type = "scse",
)
model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params / 1e6:.2f}M")


The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.


config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

Trainable parameters: 20.92M


In [ ]:
tversky_loss = smp.losses.TverskyLoss(mode="binary", alpha=0.3, beta=0.7)
focal_loss   = smp.losses.FocalLoss(mode="binary", gamma=2.0)

def combined_loss(preds, masks):
    return 0.6 * tversky_loss(preds, masks) + 0.4 * focal_loss(preds, masks)

optimizer = torch.optim.AdamW([
    {"params": model.encoder.parameters(), "lr": LR * 0.1},
    {"params": model.decoder.parameters(), "lr": LR},
    {"params": model.segmentation_head.parameters(), "lr": LR},
], weight_decay=1e-4)

scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

print("Loss, optimizer, and scheduler configured.")

Loss, optimizer, and scheduler configured.


In [ ]:
def calculate_metrics(preds, masks, threshold=0.5, eps=1e-7):
    preds = (torch.sigmoid(preds) > threshold).float()

    tp = (preds * masks).sum()
    fp = (preds * (1 - masks)).sum()
    fn = ((1 - preds) * masks).sum()
    tn = ((1 - preds) * (1 - masks)).sum()

    dice      = (2 * tp) / (2 * tp + fp + fn + eps)
    iou       = tp / (tp + fp + fn + eps)
    precision = tp / (tp + fp + eps)
    recall    = tp / (tp + fn + eps)
    f1        = dice

    return dice.item(), iou.item(), precision.item(), recall.item()

In [ ]:
def train_one_epoch(model, loader, optimizer):
    model.train()
    totals = {"loss": 0, "dice": 0, "iou": 0}

    for images, masks in loader:
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()

        outputs = model(images)
        loss    = combined_loss(outputs, masks)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        dice, iou, *_ = calculate_metrics(outputs, masks)
        totals["loss"] += loss.item()
        totals["dice"] += dice
        totals["iou"]  += iou

    n = len(loader)
    return totals["loss"]/n, totals["dice"]/n, totals["iou"]/n


def validate_one_epoch(model, loader):
    model.eval()
    totals = {"loss": 0, "dice": 0, "iou": 0, "prec": 0, "rec": 0}

    with torch.no_grad():
        for images, masks in loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            outputs = model(images)
            loss    = combined_loss(outputs, masks)
            dice, iou, prec, rec = calculate_metrics(outputs, masks)

            totals["loss"] += loss.item()
            totals["dice"] += dice
            totals["iou"]  += iou
            totals["prec"] += prec
            totals["rec"]  += rec

    n = len(loader)
    return (totals["loss"]/n, totals["dice"]/n, totals["iou"]/n,
            totals["prec"]/n, totals["rec"]/n)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

CKPT_DIR = "/content/drive/MyDrive/fracture_unetpp_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)
BEST_CKPT = f"{CKPT_DIR}/best_unetpp_efficientnetb4.pth"

try:
    ckpt = torch.load(BEST_CKPT, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    start_epoch   = ckpt["epoch"]
    history       = ckpt["history"]
    best_val_dice = ckpt["best_val_dice"]
    print(f"Resumed from epoch {start_epoch}, best Dice: {best_val_dice:.4f}")
except FileNotFoundError:
    start_epoch   = 0
    best_val_dice = 0.0
    history = {k: [] for k in ["train_loss", "val_loss", "train_dice", "val_dice",
                                "train_iou",  "val_iou",  "val_precision", "val_recall"]}
    print("No checkpoint found — starting fresh.")

Mounted at /content/drive
No checkpoint found — starting fresh.


In [ ]:
patience         = 10
epochs_no_improve = 0

for epoch in range(start_epoch, EPOCHS):
    tr_loss, tr_dice, tr_iou          = train_one_epoch(model, train_loader, optimizer)
    va_loss, va_dice, va_iou, va_p, va_r = validate_one_epoch(model, val_loader)
    scheduler.step()

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(va_loss)
    history["train_dice"].append(tr_dice)
    history["val_dice"].append(va_dice)
    history["train_iou"].append(tr_iou)
    history["val_iou"].append(va_iou)
    history["val_precision"].append(va_p)
    history["val_recall"].append(va_r)

    print(f"Epoch [{epoch+1:>2}/{EPOCHS}]  "
          f"Train Loss: {tr_loss:.4f}  Dice: {tr_dice:.4f}  IoU: {tr_iou:.4f}  |  "
          f"Val Loss: {va_loss:.4f}  Dice: {va_dice:.4f}  IoU: {va_iou:.4f}  "
          f"P: {va_p:.4f}  R: {va_r:.4f}")

    if va_dice > best_val_dice:
        best_val_dice     = va_dice
        epochs_no_improve = 0
        torch.save({
            "epoch":                epoch + 1,
            "model_state_dict":     model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "history":              history,
            "best_val_dice":        best_val_dice,
        }, BEST_CKPT)
        print(f"  ✓ Best model saved (Dice: {best_val_dice:.4f})")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"\nEarly stopping at epoch {epoch+1} (no improvement for {patience} epochs)")
            break

print(f"\nTraining complete. Best Val Dice: {best_val_dice:.4f}")

Epoch [ 1/50]  Train Loss: 0.4033  Dice: 0.3267  IoU: 0.2105  |  Val Loss: 0.3088  Dice: 0.4873  IoU: 0.3420  P: 0.5109  R: 0.5211
  ✓ Best model saved (Dice: 0.4873)
Epoch [ 2/50]  Train Loss: 0.3312  Dice: 0.4193  IoU: 0.2800  |  Val Loss: 0.2810  Dice: 0.4774  IoU: 0.3257  P: 0.3812  R: 0.7068
Epoch [ 3/50]  Train Loss: 0.3159  Dice: 0.4426  IoU: 0.2984  |  Val Loss: 0.2768  Dice: 0.5202  IoU: 0.3685  P: 0.4875  R: 0.6120
  ✓ Best model saved (Dice: 0.5202)
Epoch [ 4/50]  Train Loss: 0.3077  Dice: 0.4566  IoU: 0.3112  |  Val Loss: 0.2670  Dice: 0.5179  IoU: 0.3651  P: 0.4477  R: 0.6701
Epoch [ 5/50]  Train Loss: 0.2975  Dice: 0.4730  IoU: 0.3242  |  Val Loss: 0.2637  Dice: 0.5449  IoU: 0.3916  P: 0.5182  R: 0.6225
  ✓ Best model saved (Dice: 0.5449)
Epoch [ 6/50]  Train Loss: 0.2921  Dice: 0.4811  IoU: 0.3327  |  Val Loss: 0.2534  Dice: 0.5370  IoU: 0.3825  P: 0.4577  R: 0.6994
Epoch [ 7/50]  Train Loss: 0.2812  Dice: 0.4984  IoU: 0.3469  |  Val Loss: 0.2696  Dice: 0.5405  IoU: 0.38

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("U-Net++ EfficientNet-B4 — Training Curves", fontsize=14, fontweight="bold")

for ax, (tr_key, va_key, title, ylabel) in zip(axes, [
    ("train_loss",  "val_loss",  "Loss",       "Loss"),
    ("train_dice",  "val_dice",  "Dice Score", "Dice"),
    ("train_iou",   "val_iou",   "IoU Score",  "IoU"),
]):
    ax.plot(history[tr_key], label="Train", linewidth=2)
    ax.plot(history[va_key], label="Validation", linewidth=2, linestyle="--")
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{CKPT_DIR}/unetpp_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
ckpt = torch.load(BEST_CKPT, map_location=DEVICE)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

te_loss, te_dice, te_iou, te_prec, te_rec = validate_one_epoch(model, test_loader)

print("=" * 50)
print("U-Net++ EfficientNet-B4 — TEST SET RESULTS")
print("=" * 50)
print(f"Loss:      {te_loss:.4f}")
print(f"Dice:      {te_dice:.4f}")
print(f"IoU:       {te_iou:.4f}")
print(f"Precision: {te_prec:.4f}")
print(f"Recall:    {te_rec:.4f}")
print("=" * 50)

In [ ]:
mean = np.array([0.485, 0.456, 0.406])
std  = np.array([0.229, 0.224, 0.225])

images_b, masks_b = next(iter(test_loader))
with torch.no_grad():
    preds_b = (torch.sigmoid(model(images_b.to(DEVICE))) > 0.5).float().cpu()

fig, axes = plt.subplots(4, 3, figsize=(12, 16))
fig.suptitle("U-Net++ EfficientNet-B4 — Test Predictions", fontsize=14, fontweight="bold")

for i in range(min(4, images_b.shape[0])):
    img = images_b[i].permute(1, 2, 0).numpy()
    img = np.clip(std * img + mean, 0, 1)

    axes[i][0].imshow(img);               axes[i][0].set_title("X-ray");        axes[i][0].axis("off")
    axes[i][1].imshow(masks_b[i][0], cmap="hot");  axes[i][1].set_title("Ground Truth"); axes[i][1].axis("off")
    axes[i][2].imshow(preds_b[i][0], cmap="hot");  axes[i][2].set_title("Predicted");    axes[i][2].axis("off")

plt.tight_layout()
plt.savefig(f"{CKPT_DIR}/unetpp_predictions.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
unetpp_results = {
    "model":     "U-Net++ EfficientNet-B4",
    "dice":      round(te_dice, 4),
    "iou":       round(te_iou, 4),
    "precision": round(te_prec, 4),
    "recall":    round(te_rec, 4),
    "loss":      round(te_loss, 4),
    "img_size":  IMG_SIZE,
    "epochs":    len(history["val_dice"]),
}

import json
with open(f"{CKPT_DIR}/unetpp_results.json", "w") as f:
    json.dump(unetpp_results, f, indent=2)

print("Results saved:", unetpp_results)